In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
from itertools import combinations
from skbio.stats.distance import anosim
from skbio.stats.distance import DistanceMatrix

base_folder = '/active-data/analysis_results/chr_pla/genus'
genus_name = 'Escherichia'
random_seed = 42
model_list = [
    ("evo2_mean", "Evo2"),
    ("glm2_mean", "gLM2"),
    ("nt_mean", "Nucleotide Transformer")
]
categories = [
    'intermediate replicon',
    'typical plasmid',
    'chromosome fragment',
    'typical chromosome'
]
cat_label = [
    'Intermediate\nReplicon',
    'Typical\nPlasmid',
    'Chromosome\nFragment',
    'Typical\nChromosome'
]
cat_color = {
    'intermediate replicon':'#e74c3c',
    'typical plasmid':'#3498db',
    'chromosome fragment':'#f39c12',
    'typical chromosome':'#27ae60'
}

for model_dir_name, plot_title in model_list:
    target_dir = f'{base_folder}/embedding_vector/{genus_name}/{model_dir_name}'
    vector_dir = f"{target_dir}/random_seed_{random_seed}"
    info_file = f"{vector_dir}_samples_info.tsv"

    print(f"===== Processing {plot_title} =====")
    print(f"Metadata path: {info_file}")
    info_df = pd.read_csv(info_file, sep='\t')

    all_vectors = []
    all_labels = []
    for _, row in info_df.iterrows():
        vec_path = f"{vector_dir}/{row['sequence']}.npy"
        vec = np.load(vec_path)
        all_vectors.append(vec)
        all_labels.append(row["category-pident_90"])
    vec_mat = np.vstack(all_vectors)

    pca_2d = PCA(n_components=2, random_state=random_seed)
    vec_2d = pca_2d.fit_transform(vec_mat)
    pca_df = pd.DataFrame(vec_2d, columns=["PC1", "PC2"])
    pca_df["category-pident_90"] = all_labels
    pca_df["sequence"] = info_df["sequence"].values

    pca_df.to_csv(f"{vector_dir}_pca_2d.tsv", sep='\t', index=False)
    np.save(f"{vector_dir}_pca_2d_components.npy", pca_2d.components_)
    np.save(f"{vector_dir}_pca_explained.npy", pca_2d.explained_variance_ratio_)

    print(f"PC1 explained variance: {pca_2d.explained_variance_ratio_[0]:.4f}")
    print(f"PC2 explained variance: {pca_2d.explained_variance_ratio_[1]:.4f}")
    print(f"Total 2D explained variance: {sum(pca_2d.explained_variance_ratio_):.4f}")

    label_arr = np.array(all_labels)
    n_group = len(categories)
    r_matrix = np.zeros((n_group, n_group))
    p_matrix = np.zeros((n_group, n_group))
    result_list = []

    dist_condensed = pdist(vec_mat, metric="euclidean")
    full_dist_matrix = squareform(dist_condensed)

    for g1, g2 in combinations(categories, 2):
        idx1 = categories.index(g1)
        idx2 = categories.index(g2)
        keep_mask = (label_arr == g1) | (label_arr == g2)
        keep_idx = np.where(keep_mask)[0]

        sub_dist = full_dist_matrix[np.ix_(keep_idx, keep_idx)]
        sub_ids = [str(i) for i in range(len(keep_idx))]
        sub_dm = DistanceMatrix(sub_dist, ids=sub_ids)
        sub_group_label = label_arr[keep_idx].tolist()

        np.random.seed(random_seed)
        anosim_out = anosim(sub_dm, sub_group_label, permutations=999)

        r = anosim_out["test statistic"]
        p = anosim_out["p-value"]

        r_matrix[idx1, idx2] = r
        r_matrix[idx2, idx1] = r
        p_matrix[idx1, idx2] = p
        p_matrix[idx2, idx1] = p
        result_list.append({
            "group1":g1,
            "group2":g2,
            "anosim_R_highdim_euclidean":r,
            "anosim_pvalue":p
        })

    anosim_df = pd.DataFrame(result_list)
    anosim_df.to_csv(f"{vector_dir}_anosim_highdim_euclidean_result.tsv", sep="\t", index=False)
    print(f"=== {plot_title} Finished ===\n")
    print(anosim_df.round(4))

print("All computation finished.")

/home/zhongshitong/miniconda3/envs/R-env/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


===== Processing Evo2 =====
Metadata path: /active-data/analysis_results/chr_pla/genus/embedding_vector/Escherichia/evo2_mean/random_seed_42_samples_info.tsv
PC1 explained variance: 0.8947
PC2 explained variance: 0.0975
Total 2D explained variance: 0.9922
=== Evo2 Finished ===

                  group1               group2  anosim_R_highdim_euclidean  \
0  intermediate replicon      typical plasmid                      0.0533   
1  intermediate replicon  chromosome fragment                      0.0157   
2  intermediate replicon   typical chromosome                      0.1409   
3        typical plasmid  chromosome fragment                      0.0691   
4        typical plasmid   typical chromosome                      0.1131   
5    chromosome fragment   typical chromosome                      0.0518   

   anosim_pvalue  
0          0.001  
1          0.070  
2          0.001  
3          0.001  
4          0.001  
5          0.001  
===== Processing gLM2 =====
Metadata path: /acti